# 🧠 Phase-3 Second-Order Growth — WikiText-2 Graduation Run (Experiment 61)

**Does a row-centered SPPMI *second-order* operator grow _paradigmatic_ structure on real text that the first-order co-occurrence centroid failed at (Report 121)?**

This notebook runs `experiments/61_phase3_second_order_growth.py` on **WikiText-2** on a GPU.

- It grows a **separate paradigmatic codebook `G`** — the validated 055–058 recall **floor is untouched by construction**.
- **B′** (primary): `S' = rownorm(SPPMI @ SPPMIᵀ) − rowmean`, `centroid = S' @ G`. Row-centering removes the near-uniform leading eigenvector that otherwise collapses `G` (the red-team grill's load-bearing fix).
- Controls: **B** (uncentered — expected to collapse/leak) and **A** (first-order `P@Pᵀ`).
- **4-condition gate** (all must hold): ① demeaned matched-specificity headline CI > 0 · ② collapse floor `d_eff_end/init ≥ 0.5 ∧ off-max < 0.99 ∧ mean-off ≤ ceiling` · ③ decorrelation `corr(log cooc, drift) CI-hi < 0.15` · ④ gauge-validity `corr(S_real, S_shuffle) < 0.40`.
- **Pre-registered null disposition:** if no `(k, α)` cell clears the headline **and** the collapse floor → **NULL** → §10 oracles (SVD-of-SPPMI + FHRR-port) → latent-layer fork. *Do not scale to rescue a null.*

Spec: `notes/emergent-codebook/phase-3-second-order-growth-precommit.md` · ratified gate: `notes/emergent-codebook/phase-3-structure-gate-3b-design.md`.

## 0 · Set the runtime to GPU
**Runtime → Change runtime type → T4 GPU** (or better) before running.

In [ ]:
!nvidia-smi -L || echo "⚠️  No GPU detected — set Runtime → Change runtime type → GPU." 

In [ ]:
# If the repo is PRIVATE, paste a GitHub token (Settings → Developer settings → Tokens, `repo` scope).
# Leave it "" if the repo is public.
GITHUB_TOKEN = ""  #@param {type:"string"}
BRANCH = "consolidation/role-structure"
_host = "github.com/Dypatterson/Neuro-AI.git"
clone_url = f"https://{GITHUB_TOKEN}@{_host}" if GITHUB_TOKEN else f"https://{_host}"

%cd /content
!rm -rf Neuro-AI
!git clone --depth 1 --branch {BRANCH} {clone_url}
%cd /content/Neuro-AI
!git log --oneline -1
!pip -q install datasets

In [ ]:
# Pre-warm the WikiText-2 cache. (Pure data load — does NOT touch CUDA, keeping the GPU
# clean for the experiment subprocess, per the project's Colab gotcha.)
from datasets import load_dataset
load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
print("✅ WikiText-2 cached.")

## 1 · GPU sanity check — does B′ still thread the needle?
A ~30-second **planted** run on the GPU (king/queen share contexts but never co-occur). Confirms the CUDA growth path works and B′ lifts the planted pair *specifically* without collapse — **before** the expensive headline run.

In [ ]:
!PYTHONPATH=src python experiments/61_phase3_second_order_growth.py \
  --corpus-source synthetic_planted --variants B_prime,B,A \
  --D 512 --W 6 --epochs 8 --alpha0 0.05 --alpha-decay 0.9 --seeds 3 \
  --k-grid 4,6 --device cuda --out reports/_exp61_planted_gpu.json

In [ ]:
import json
S = json.load(open("reports/_exp61_planted_gpu.json"))
for v in ["B_prime", "B", "A"]:
    for r in S["grid_results"]:
        if r["variant"] == v and r.get("planted_probe"):
            p = r["planted_probe"]
            print(f"{v:8} k={r['k']} α0={r['alpha0']}: "
                  f"king/queen={p['drift_king_queen']:+.3f}  distractor={p['drift_king_distractor']:+.3f}  "
                  f"random={p['drift_random_nontarget_mean']:+.3f}  d_eff_ratio={p['d_eff_ratio']:.2f}  "
                  f"THREADS_NEEDLE={p['THREADS_NEEDLE']}")
print("\n✅ Expect: B_prime THREADS_NEEDLE=True (king/queen ≫ distractor & random, d_eff retained);")
print("   B leaks/collapses; A ≈ B_prime on this frequency-controlled toy (PMI's edge is real-text-only).")

## 2 · The headline run — WikiText-2 (the graduation test)
B′ + B + A · **k pinned by SPPMI density** · **α swept** `{0.05, 0.1, 0.2, 0.3}` (the grill found a narrow non-collapsing band) · **SimLex-999 ≥ 5.0** paradigmatic pairs · **5 seeds** · hierarchical seed×pair bootstrap. A few minutes on a T4.

In [ ]:
!PYTHONPATH=src python experiments/61_phase3_second_order_growth.py \
  --corpus-source wikitext --wikitext-name wikitext-2-raw-v1 \
  --pair-source simlex --simlex-min-sim 5.0 \
  --variants B_prime,B,A \
  --D 1024 --W 6 --epochs 20 \
  --alpha-grid 0.05,0.1,0.2,0.3 \
  --seeds 5 --max-vocab 2000 --paradigmatic-max-cooc 2 \
  --device cuda \
  --out reports/_exp61_wikitext_headline.json

## 3 · Results — the 4-condition gate

In [ ]:
import json
S = json.load(open("reports/_exp61_wikitext_headline.json"))
cfg = S["config"]
print("VERDICT:", S["verdict"])
print("ANY_VARIANT_PASS:", S["ANY_VARIANT_PASS"])
print(f"k pinned by density = {cfg['k_chosen_by_density']} (density {cfg['density']:.3f}, target {cfg['density_target']})")
print("SimLex:", S["pair_source"], "| fallback pairs used:", S["fallback_pairs_used"])
print()
cols = (f"{'variant':7} {'k':>3} {'α0':>5} {'headline real-shuf (95% CI)':>32} "
        f"{'>0':>4} {'collapse':>8} {'d_eff_r':>7} {'corr<.15':>8} {'gauge':>6} {'PASS':>5}")
print(cols); print("-" * len(cols))
for r in S["grid_results"]:
    h, cf = r["HEADLINE_demeaned_matched_real_minus_shuffle"], r["collapse_floor"]
    dc, gv = r["decorrelation"], r["gauge_validity"]
    hm, ci = h["hierarchical_mean"], h["hierarchical_ci"]
    cis = f"{hm:+.4f} [{ci[0]:+.4f},{ci[1]:+.4f}]" if hm == hm and ci[0] == ci[0] else "nan"
    dr = cf.get("d_eff_ratio_real", float("nan"))
    print(f"{r['variant']:7} {r['k']:>3} {r['alpha0']:>5} {cis:>32} "
          f"{str(h['CI_gt_0']):>4} {str(cf['COLLAPSE_OK']):>8} {dr:>7.3f} "
          f"{str(dc['ci_hi_lt_0p15']):>8} {str(gv['VALID']):>6} {str(r['VARIANT_PASS']):>5}")
bp = S.get("B_prime_headline_cell")
if bp:
    print(f"\nB′ headline cell: k={bp['k']} α0={bp['alpha0']} → PASS={bp['VARIANT_PASS']}, "
          f"corr(log cooc,drift)={bp['decorrelation']['corr_logcooc_drift_point']}")
print("\n→ PASS = first paradigmatic signal (then run the §10.1 mixing-feasibility pre-check — still NOT a graduation claim).")
print("→ NULL = per the pre-registered §10 disposition: run the SVD-of-SPPMI + FHRR-port oracles; do NOT scale vocab/epochs.")

## 4 · Save results
Copies the JSON to Drive and downloads it. Bring the headline JSON back to the repo for the writeup (Report 122).

In [ ]:
from google.colab import drive, files
import shutil, pathlib
drive.mount("/content/drive")
dst = pathlib.Path("/content/drive/MyDrive/neuro-ai/results/exp61")
dst.mkdir(parents=True, exist_ok=True)
for f in ["reports/_exp61_wikitext_headline.json", "reports/_exp61_planted_gpu.json"]:
    p = pathlib.Path(f)
    if p.exists():
        shutil.copy(p, dst / p.name)
        files.download(f)
print("✅ saved to", dst)